# Unit 6 practical: fine-tune a small open model with LoRA
> **Status: written but NOT executed** while the course was authored (no GPU, no model, no `peft` library on the authoring machine).
> The base model exists on the Hugging Face hub (checked). The attention module names (`q_proj`, `v_proj`) are **assumed**: print the model to confirm them.
> Run on Google Colab with a T4 GPU. If a cell fails, a library version has probably changed.

You will: (1) build a small SFT data set and **split it by hash**, (2) tokenize with a chat template and **mask the prompt** in the loss, (3) attach **LoRA** adapters, (4) train, (5) check the result against the base model and run the **forgetting regression test**, (6) **merge** the adapter.

In [ ]:
!pip -q install transformers peft accelerate
!wget -q https://krunalpanchal1995.github.io/banu.github.io/downloads/u6/u6-code.zip -O u6.zip && unzip -q -o u6.zip
import sys; sys.path.insert(0, "u6-code/codes/adapt/adapt_np"); import adapt_np as A
import torch, random, json

## 1. A tiny SFT data set (replace with your own domain)
The task: always answer in a fixed format `ANSWER: <short answer> | CONFIDENCE: <low/medium/high>`. This is a *behaviour* change, which is what fine-tuning is good at.

In [ ]:
facts = [("capital of France", "Paris"), ("capital of Japan", "Tokyo"), ("capital of Italy", "Rome"), ("capital of Spain", "Madrid"),
         ("capital of Kenya", "Nairobi"), ("capital of Canada", "Ottawa"), ("capital of Egypt", "Cairo"), ("capital of India", "New Delhi"),
         ("capital of Brazil", "Brasilia"), ("capital of Peru", "Lima"), ("capital of Norway", "Oslo"), ("capital of Ghana", "Accra")]
records = [{"instruction": f"What is the {q}?", "response": f"ANSWER: {a} | CONFIDENCE: high"} for q, a in facts]
for r in records: assert A.validate_sft_record(r) == []
print("near duplicates:", A.near_duplicates([r["instruction"] for r in records], 0.8))
train, val = A.split_by_hash(records, "instruction", val_frac=0.25)
print(len(train), "train /", len(val), "validation")

## 2. Tokenize with a chat template and mask the prompt (loss on the response only)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
name = "Qwen/Qwen2.5-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(name)
def encode(rec):
    prompt = tok.apply_chat_template([{"role": "user", "content": rec["instruction"]}], tokenize=False, add_generation_prompt=True)
    p_ids = tok(prompt, add_special_tokens=False)["input_ids"]; r_ids = tok(rec["response"] + tok.eos_token, add_special_tokens=False)["input_ids"]
    ids = p_ids + r_ids
    labels = [-100] * len(p_ids) + r_ids                    # -100 = ignored by the loss: the prompt is masked out (Equation 1)
    return torch.tensor([ids]), torch.tensor([labels])
print(encode(train[0])[1])

## 3. Attach LoRA adapters (only A and B are trainable)

In [ ]:
from peft import LoraConfig, get_peft_model
model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.float32, device_map="auto")
print([n for n, _ in model.named_modules() if n.endswith("q_proj") or n.endswith("v_proj")][:4])       # confirm the module names
cfg = LoraConfig(r=8, lora_alpha=16, target_modules=["q_proj", "v_proj"], lora_dropout=0.0, task_type="CAUSAL_LM")
model = get_peft_model(model, cfg)
model.print_trainable_parameters()                          # compare with the LoRA calculator on the course site

## 4. Train, tracking training and validation loss

In [ ]:
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=2e-4)
def loss_on(rec):
    ids, labels = encode(rec); return model(input_ids=ids.to(model.device), labels=labels.to(model.device)).loss
hist = []
for epoch in range(8):
    random.shuffle(train); model.train(); tl = 0
    for rec in train:
        loss = loss_on(rec); loss.backward(); opt.step(); opt.zero_grad(); tl += loss.item()
    model.eval()
    with torch.no_grad(): vl = sum(loss_on(r).item() for r in val) / len(val)
    hist.append((tl / len(train), vl)); print(f"epoch {epoch}: train {tl/len(train):.3f}  validation {vl:.3f}")

## 5. Did it work? And did it forget? (Section 3.4.2 of the PDF)

In [ ]:
def generate(text, adapter=True):
    ids = tok.apply_chat_template([{"role": "user", "content": text}], add_generation_prompt=True, return_tensors="pt").to(model.device)
    ctx = model.disable_adapter() if not adapter else torch.no_grad()
    with ctx:
        out = model.generate(ids, max_new_tokens=40, do_sample=False)
    return tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True)
print("tuned   :", generate("What is the capital of Germany?"))                  # a held-out country: does the format transfer?
print("base    :", generate("What is the capital of Germany?", adapter=False))
# Regression test on an OLD ability: a few general questions the base model answers well. Compare before/after by eye or with a metric.
general = ["Explain what a prime number is in one sentence.", "Translate 'good morning' into Spanish.", "What is 17 + 25?"]
for q in general: print(q, "\n  tuned:", generate(q), "\n  base :", generate(q, adapter=False))

## 6. Merge for serving and save
`merge_and_unload()` folds $W_0 + \frac{\alpha}{r}BA$ into the weights (Section 3.2.2): no extra latency at inference. Save the small adapter separately if you want to keep the base model shared.

In [ ]:
model.save_pretrained("my-lora-adapter")         # a few MB
merged = model.merge_and_unload()
print(type(merged))

## Your turn
1. Increase the number of examples to 100+ and watch the validation loss. At what point does it stop improving (Figure 6 of the PDF)?
2. Change the rank to 1, 4, 32 and compare the validation loss and the regression test. Does a higher rank forget more or less (Section 3.1)?
3. Mix 20% general examples into the training data (replay) and re-run the regression test.
4. Turn off the prompt masking (`labels = ids`) and compare validation loss on the response only.